# Introduction
This notebook intends to understand the data in the birdCLEF_2025 dataset and create some simple classifier models for recognizing species by their sound.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from IPython.display import Audio
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
                             roc_curve, auc, precision_recall_curve, average_precision_score)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import label_binarize
import time
from tqdm import tqdm
from sklearn.utils import resample
from matplotlib.colors import ListedColormap
from imblearn.over_sampling import SMOTE
from collections import Counter
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import label_binarize


In [ ]:
# Load data

INPUT_PATH = os.path.join('birdclef-2025')
TRAIN_AUDIO_PATH = os.path.join(INPUT_PATH, 'train_audio')
TEST_SOUNDSCAPES_PATH = os.path.join(INPUT_PATH, 'test_soundscapes')
TRAIN_SOUNDSCAPES_PATH = os.path.join(INPUT_PATH, 'train_soundscapes')

# Load data
taxonomy = pd.read_csv(os.path.join(INPUT_PATH, 'taxonomy.csv'))
train_meta = pd.read_csv(os.path.join(INPUT_PATH, 'train.csv'))
# recording_locations = pd.read_csv(os.path.join(INPUT_PATH, 'recordming_location.txt'))

## Understanding our data

In [ ]:
print(train_meta.dtypes)
train_meta

In [ ]:
print(taxonomy.dtypes)
taxonomy # we see that taxonomy allow us to match 'primarry_lables' of train.csv to the species

### Preprocessing

In [ ]:
# check for duplicate rows
print(f"number of duplicate rows: {train_meta.duplicated().sum()}")

# check if secondary_label and type are empty
print("secondary_labels is empty: " + str((train_meta['secondary_labels'] == '[\'\']').all()))
print("type is empty: " + str((train_meta['type'] == "[\'\']").all()))

# check for NaN values
print(f'number of na values by column \n{train_meta.isna().sum()}')

# remove samples with unavailable locations
train_meta.dropna(inplace=True)

# remove irrelevant columns 
train_meta.drop(['rating', 'collection', 'author', 'license', 'scientific_name', 'url', 'secondary_labels', 'type'], axis=1, inplace=True)

# link the filenames to our dataset and turn secondary_labels and columns to lists
def preprocess_train_meta(df):
    # df['secondary_labels'] = df['secondary_labels'].apply(lambda x: re.findall(r"'(\w+)'", x))
    # df['type'] = df['type'].apply(lambda x: re.findall(r"'(\w+)'", x))
    df['file_path'] = df.apply(lambda row: os.path.join(TRAIN_AUDIO_PATH, row['filename']), axis=1)
    return df

train_meta = preprocess_train_meta(train_meta)

# Merge class_name from taxonomy into train_meta using primary_label as key
train_meta = train_meta.merge(taxonomy[['primary_label', 'class_name']], on='primary_label', how='left')

train_meta

In [ ]:
# Create side-by-side plots
fig, axs = plt.subplots(1, 2, figsize=(16, 6))

# Plot for taxonomy
sns.countplot(
    data=taxonomy,
    y='class_name',
    order=taxonomy['class_name'].value_counts().index,
    ax=axs[0]
)
axs[0].set_title('Class Distribution in Taxonomy')
axs[0].set_xlabel('Count')
axs[0].set_ylabel('Class Name')

# Plot for train_meta
sns.countplot(
    data=train_meta,
    y='class_name',
    order=taxonomy['class_name'].value_counts().index,  # Same order for consistency
    ax=axs[1]
)
axs[1].set_title('Class Distribution in Train Meta')
axs[1].set_xlabel('Count')
axs[1].set_ylabel('')

plt.tight_layout()
plt.show()

We can see that there is a large class imbalance in the dataset. 

In [ ]:
label_distribution = train_meta['primary_label'].value_counts()

print(label_distribution.value_counts(), "\n")
print(f"Unique primary labels in train_meta: {label_distribution.nunique()}")\

plt.figure(figsize=(10, 6))
sns.barplot(x=label_distribution.index, y=label_distribution.values)
plt.title('Distribution of Primary Labels')
plt.xlabel('Primary Label')
plt.ylabel('Count')
plt.xticks([])
plt.tight_layout()
plt.show()

As expected, the imbalance is also significant for primary labels. 

Another problem is the large computational time for processing many audio files, so, we first decide to filter audio files based on their duration.

In [ ]:
# sample
sampled_meta = train_meta.sample(n=200, random_state=42)

# function to get a file's duration
def get_duration(file_path):
    try:
        y, sr = librosa.load(file_path, sr=None)  # Correct variable
        duration = librosa.get_duration(y=y, sr=sr)
        return duration
    except Exception as e:
        print(f"Could not load {file_path}: {e}")
        return None  # Use None to signify failure

# Collect durations
durations = []
for path in tqdm(sampled_meta['file_path'], desc="Measuring durations"):
    durations.append(get_duration(path))

# Plot the distribution
plt.figure(figsize=(20, 6))
plt.hist(durations, bins=50, color='skyblue', edgecolor='black')
plt.title("Distribution of Audio File Durations")
plt.xlabel("Duration (seconds)")
plt.ylabel("Number of Files")
plt.grid(True)
plt.show()

We decide to keep files between 10 and 20 seconds long.

In [ ]:
tqdm.pandas()

def filter_audio_duration(df, lower_bound, upper_bound):
    df['duration'] = df['file_path'].progress_apply(get_duration)
    df_filtered = df[
        (df['duration'] >= lower_bound) & (df['duration'] <= upper_bound)
    ].copy()
    return df_filtered

print(f"Size before filtering: {len(train_meta)}")

if not os.path.exists('filtered_duration.csv'):
    train_meta_filtered = filter_audio_duration(train_meta, 5, 20)
    train_meta_filtered.to_csv('filtered_duration.csv', index=False)
else:
    train_meta_filtered = pd.read_csv('filtered_duration.csv')
print(f"Size after filtering: {len(train_meta_filtered)}\n")


# Plot the filtered durations
plt.figure(figsize=(10, 6))
plt.hist(train_meta_filtered['duration'], bins=50, alpha=0.6, label='Train', color='skyblue', edgecolor='black')
# plt.hist(test_df_filtered['duration'], bins=50, alpha=0.6, label='Test', color='salmon', edgecolor='black')
# plt.hist(val_df_filtered['duration'], bins=50, alpha=0.6, label='Validation', color='yellow', edgecolor='black')
plt.title("Distribution of Audio File Durations (5 to 15 seconds)")
plt.xlabel("Duration (seconds)")
plt.ylabel("Number of Files")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
label_distribution = train_meta_filtered['primary_label'].value_counts()

# print(label_distribution.value_counts(), "\n")
print(f"Unique primary labels in train_meta_filtered: \n{label_distribution}\n")
print(f"Unique primary labels in train_meta_filtered: {label_distribution.nunique()}")

plt.figure(figsize=(10, 6))
sns.barplot(x=label_distribution.index, y=label_distribution.values)
plt.title('Distribution of Primary Labels')
plt.xlabel('Primary Label')
plt.ylabel('Count')
plt.xticks([])
plt.tight_layout()
plt.show()

We will also filter out species with 10 or less recordings, thus avoiding outliers and enabling stratifying.

In [ ]:
label_counts = train_meta_filtered['primary_label'].value_counts()
valid_labels = label_counts[label_counts >= 10].index
train_meta_filtered = train_meta_filtered[train_meta_filtered['primary_label'].isin(valid_labels)].reset_index(drop=True).copy()

label_distribution = train_meta_filtered['primary_label'].value_counts()
print(f"Unique primary labels in train_meta_filtered: {label_distribution.nunique()}")

At this point we would like to split the data into a training set, test set and validation set, while stratifying, so that proportions are not lost for testing. However, since we've decided to use the sound files and locations for classifying, we need to aggregate the data first.

### Feature extraction

Based on the EDA and additional research, we decided to use Mel-Frequency Cepstral Coefficients (MFCCs) to train our model.

In [ ]:
# lets explore an audio file
audio_examples = train_meta.sample(1)
file_path = audio_examples['file_path'].tolist()[0]
title = audio_examples['primary_label'].tolist()[0]


Audio(file_path)
y, sr = librosa.load(file_path)
# Plot waveform
plt.figure(figsize=(14, 4))
plt.subplot(1, 2, 1)
librosa.display.waveshow(y, sr=sr)
plt.title("Waveform: " + title )

# Plot spectrogram
plt.subplot(1, 2, 2)
D = librosa.stft(y)
S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)
librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='log')
plt.title(f'Spectrogram: {title}')

plt.show()

In [ ]:
# Feature Extraction (Simple MFCC)
def extract_mfcc(file_path, sr=22050, n_mfcc=20):
    """Extracts MFCC features from an audio file."""
    try:
        y, sr = librosa.load(file_path, sr=sr)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        mfccs_processed = np.mean(mfccs.T, axis=0)  # Average across time
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None
    return mfccs_processed

# Example MFCC extraction
example_file = train_meta['file_path'].iloc[0]
mfccs = extract_mfcc(example_file)
print("\nMFCC Features Example:\n", mfccs)

We extract the MFCC features, as well as the locations and aggregate them.

In [ ]:
import os

# Feature Extraction (Simple MFCC)
def extract_mfcc(file_path, sr=22050, n_mfcc=20):
    """Extracts MFCC features from an audio file."""
    try:
        y, sr = librosa.load(file_path, sr=sr)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        mfccs_processed = np.mean(mfccs.T, axis=0)  # Average across time
        return mfccs_processed
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None
    

if not os.path.exists('all_features.npy'):
    # Extract MFCC features
    features = []
    labels = []
    valid_indices = []

    for idx, row in tqdm(train_meta_filtered.iterrows(), total=len(train_meta_filtered), desc="Extracting train MFCCs"):
        mfccs = extract_mfcc(row['file_path'])
        if mfccs is not None:
            features.append(mfccs)
            labels.append(row['primary_label'])
            valid_indices.append(idx)

    mfccs = np.array(features)

    # Extract locations
    locations = train_meta_filtered.loc[valid_indices, ['longitude', 'latitude']].values
    # Concatenate audio and location features
    all_features = np.hstack([mfccs, locations])



    np.save('mfccs.npy', mfccs)
    np.save('locations.npy', np.array(locations))
    np.save('labels.npy', np.array(labels))
    np.save('all_features.npy', np.array(all_features))
else:
    print("all_features.npy already exists. Skipping feature extraction.")
    # Load features
    all_features = np.load('all_features.npy')
    mfccs = np.load('mfccs.npy')
    locations = np.load('locations.npy')
    labels = np.load('labels.npy')





In [ ]:
le = LabelEncoder()
labels_encoded = le.fit_transform(labels)

np.save('labels_encoded.npy', np.array(labels_encoded))

In [ ]:
# Feature Scaling
scaler = StandardScaler()
all_features_scaled = scaler.fit_transform(all_features)
mfccs_scaled = scaler.fit_transform(mfccs)

np.save('all_features_scaled.npy', all_features_scaled)
np.save('mfccs_scaled.npy', mfccs_scaled)

In [ ]:
print(len(all_features_scaled))
print(len(mfccs_scaled))
print(len(labels_encoded))

### Splitting and sampling functions

In [ ]:
X = np.load('mfccs_scaled.npy')
X_all = np.load('all_features_scaled.npy')
y = np.load('labels_encoded.npy')

In [ ]:
def split(X, y, test_size=0.2, validation=False, val_size=0.25):
    # train/test split
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )

    # test/val split
    if validation:
        X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=val_size, random_state=42, stratify=y_train_full)
        return X_train, y_train, X_test, y_test, X_val, y_val
    else:
        return X_train_full, y_train_full, X_test, y_test, None, None


In [ ]:
def sample(X, y, sample_fraction=0.5, random_state=42):
    X_sample, _, y_sample, _ = train_test_split(
        X, y,
        train_size=sample_fraction,
        stratify=y,
        random_state=random_state
    )
    return X_sample, y_sample

In [ ]:
def remove_classes_under(X, y, min_instances=20):
    class_counts = Counter(y)
    
    # Get valid classes
    valid_classes = {cls for cls, count in class_counts.items() if count >= min_instances}
    
    # Create mask for valid samples
    mask = np.array([label in valid_classes for label in y])
    
    # Filter X and y
    X_filtered = X[mask]
    y_filtered = y[mask]

    return X_filtered, y_filtered

## Training Models
- SVM
- Naive Bayes
- KNN
- Random Forest
- 

In [ ]:
# models eval function


def evaluate_model(model, X_test, y_test, model_name="Model", class_labels=None):
    """
    Evaluates a trained classifier, printing metrics and plotting visualizations.

    This function handles both models that can and cannot produce probability scores.

    Parameters:
    - model: The trained classifier object.
    - X_test (array-like): The test feature data.
    - y_test (array-like): The true test labels.
    - model_name (str): The name of the model for titles and printouts.
    - class_labels (list of str, optional): The original string names of the classes
      for more readable plot labels. If None, integer labels are used.
    """
    print(f"\n{'='*20} Evaluating {model_name} {'='*20}")

    # --- 1. Basic Metrics from Predictions ---
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted', zero_division=0)
    
    print(f"Accuracy: {acc:.4f}")
    print(f"Weighted Precision: {prec:.4f}")
    print(f"Weighted Recall: {rec:.4f}")
    print(f"Weighted F1-score: {f1:.4f}\n")
    
    # --- 2. Confusion Matrix ---
    cm = confusion_matrix(y_test, y_pred)
    fig_cm, ax_cm = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax_cm, 
                xticklabels=class_labels if class_labels is not None else 'auto',
                yticklabels=class_labels if class_labels is not None else 'auto')
    ax_cm.set_title(f"{model_name} Confusion Matrix")
    ax_cm.set_xlabel("Predicted Label")
    ax_cm.set_ylabel("True Label")
    plt.show()

    # --- 3. Probability-based Curves (ROC and Precision-Recall) ---
    # Check if the model has the 'predict_proba' method
    if hasattr(model, "predict_proba"):
        
        y_score = model.predict_proba(X_test)
        
        # We need to binarize the labels for multiclass plotting
        n_classes = len(np.unique(y_test))
        y_test_binarized = label_binarize(y_test, classes=range(n_classes))

        # Plot ROC Curve
        fig_roc, ax_roc = plt.subplots(figsize=(10, 8))
        RocCurveDisplay.from_predictions(
            y_test_binarized.ravel(), y_score.ravel(), name="Micro-average ROC",
            color="deeppink", linestyle=":", ax=ax_roc)
        ax_roc.set_title(f"{model_name} ROC Curve")
        ax_roc.grid(True)
        plt.show()

        # Plot Precision-Recall Curve
        fig_pr, ax_pr = plt.subplots(figsize=(10, 8))
        PrecisionRecallDisplay.from_predictions(
            y_test_binarized.ravel(), y_score.ravel(), name="Micro-average PR",
            color="navy", linestyle=":", ax=ax_pr)
        ax_pr.set_title(f"{model_name} Precision-Recall Curve")
        ax_pr.grid(True)
        plt.show()
    else:
        print("\n --- model dose not support probability predictions --- \n")

In [ ]:
    # Sample and clean
X_sampled, y_sampled = sample(X, y, 0.1)
X_sampled, y_sampled = remove_classes_under(X_sampled, y_sampled, min_instances=20)

### SVM

In [ ]:
# Define helper functions
def make_meshgrid(x, y, h=0.1):
    x_min, x_max = x.min() - 1, x.max() + 1
    y_min, y_max = y.min() - 1, y.max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    return xx, yy

def plot_contours(ax, clf, xx, yy, **params):
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    return ax.contourf(xx, yy, Z, **params)

In [ ]:
def train_SVM_model(X, y, kernel_list=['linear', 'poly', 'rbf', 'sigmoid']):
    print("--- Training XGBoost Model ---")


    print(f"Using a sample of {X.shape[0]} instances with {len(np.unique(y))} classes.")

    # Split
    X_train, y_train, X_test, y_test, X_val, y_val = split(X, y)

    # Resample
    print("Applying SMOTE to the training data...")    
    smote = SMOTE(random_state=42, k_neighbors=2)
    X_train, y_train = smote.fit_resample(X_train, y_train)

    # fig_cm, axes_cm = plt.subplots(1, len(kernel_list), figsize=(5 * len(kernel_list), 5))

    for idx, kernel in enumerate(kernel_list):
        clf = SVC(kernel=kernel, degree=2, class_weight='balanced')

        # Train
        start_train = time.time()
        clf.fit(X_train, y_train)
        print(f"{kernel} train time: {time.time() - start_train:.2f}s")

        evaluate_model(clf, X_test, y_test, 
               model_name="Support Vector Machine")

train_SVM_model(X_sampled, y_sampled)

### Naive Bayes

In [ ]:
def train_GNB_model(X, y):

    print("--- Training NB Model ---")

    # Sample and clean
    X_sampled, y_sampled = X, y
    print(f"Using a sample of {X_sampled.shape[0]} instances with {len(np.unique(y_sampled))} classes.")

    # Split
    X_train, y_train, X_test, y_test, X_val, y_val = split(X_sampled, y_sampled)

    # Resample
    print("Applying SMOTE to the training data...")
    smote = SMOTE(random_state=42, k_neighbors=2)
    X_train, y_train = smote.fit_resample(X_train, y_train)

    # Train model
    print("\nTraining the model...")
    model = GaussianNB()
    start_train = time.time()
    model.fit(X_train, y_train)
    print(f"GNB train time: {time.time() - start_train:.2f}s")

    # 6. Make predictions and evaluate performance
    evaluate_model(model, X_test, y_test, 
               model_name="Naive Bayes")

train_GNB_model(X_sampled, y_sampled)


### KNN

In [ ]:
def train_KNN_model(X, y):

    print("--- Training KNN Model ---")

    # Sample and clean
    X_sampled, y_sampled = X, y
    print(f"Using a sample of {X_sampled.shape[0]} instances with {len(np.unique(y_sampled))} classes.")

    # Split
    X_train, y_train, X_test, y_test, X_val, y_val = split(X_sampled, y_sampled)

    # Resample
    print("Applying SMOTE to the training data...")
    smote = SMOTE(random_state=42, k_neighbors=2)
    X_train, y_train = smote.fit_resample(X_train, y_train)
    print(f"Training data shape after SMOTE: {X_train.shape}")

    # Train model
    model = KNeighborsClassifier(n_neighbors=3, p=1)
    start_train = time.time()
    model.fit(X_train, y_train)
    print(f"KNN train time: {time.time() - start_train:.2f}s")

    # 6. Evaluate the model using our universal function
    evaluate_model(model, X_test, y_test, 
                   model_name="K Nearest Neighbours")

train_KNN_model(X_sampled, y_sampled)

### Random Forest

In [ ]:


def train_XGB_model(X, y):
    """
    Trains, evaluates, and visualizes a simple XGBoost model,
    """
    print("--- Training XGBoost Model ---")

    #  Sample and clean the data

    print(f"Using a sample of {X.shape[0]} instances with {len(np.unique(y))} classes.")



    local_le = LabelEncoder()
    y_relabled = local_le.fit_transform(y)
    
    
    #  Split data into training and testing sets
    X_train, y_train, X_test, y_test, _, _ = split(X, y_relabled)

    print(f"Training data shape: {X_train.shape}")

    # Handle class imbalance
    print("Applying SMOTE to the training data...")
    min_class_count = np.min(np.unique(y_train, return_counts=True)[1])
    k_neighbors = min(2, min_class_count - 1) if min_class_count > 1 else 1
    smote = SMOTE(random_state=42, k_neighbors=k_neighbors)
    X_train, y_train = smote.fit_resample(X_train, y_train)

    n_classes = len(np.unique(y_train))
    print(f"Training data shape after SMOTE: {X_train.shape}")
    print(f"Number of classes: {n_classes}")

    # Define and train the XGBoost model
    model = xgb.XGBClassifier(
        objective="multi:softprob",
        random_state=42,
    )

    print("\nTraining the model...")
    
    
    model.fit(X_train, y_train)
    
    # Make predictions and evaluate performance
    evaluate_model(model, X_test, y_test, model_name="XGBoost Classifier")

    params = {
            'min_child_weight': [1, 5, 10],
            'gamma': [0.5, 1, 1.5, 2, 5],
            'subsample': [0.6, 0.8, 1.0],
            'colsample_bytree': [0.6, 0.8, 1.0],
            'max_depth': [3, 4, 5]
    }

    # Perform hyperparameter tuning using GridSearchCV
    print("\nPerforming hyperparameter tuning...")
    grid_search = GridSearchCV(
        model,
        param_grid=params,
        scoring='accuracy',
        cv=3,
        verbose=1,
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)

    # Best model
    best_model = grid_search.best_estimator_
    print("Best parameters:", grid_search.best_params_)

    # Evaluate on test set
    y_pred = best_model.predict(X_test)
    print("Test accuracy:", accuracy_score(y_test, y_pred))

    evaluate_model(best_model, X_test, y_test, model_name="Best XGBoost Classifier")

    

# Run the updated function
train_XGB_model(X_sampled, y_sampled)

### Perceptron

In [ ]:
from sklearn.linear_model import  Perceptron

def train_Perceptron_model(X, y):
    """
    Trains, evaluates, and visualizes a Perceptron model.
    """
    print("--- Training Perceptron Model ---")

    # Sample and clean the data
    X_sampled, y_sampled = X, y
    print(f"Using a sample of {X_sampled.shape[0]} instances with {len(np.unique(y_sampled))} classes.")

    
    # Split data into training and testing sets
    X_train, y_train, X_test, y_test, _, _ = split(X_sampled, y_sampled)

    # Handle class imbalance
    print("Applying SMOTE to the training data...")
    min_class_count = np.min(np.unique(y_train, return_counts=True)[1])
    k_neighbors = min(2, min_class_count - 1) if min_class_count > 1 else 1
    smote = SMOTE(random_state=42, k_neighbors=k_neighbors)
    X_train, y_train = smote.fit_resample(X_train, y_train)
    

    print(f"Training data shape after SMOTE: {X_train.shape}")


    # 5. Define and train the Logistic Regression model
    model = Perceptron(random_state=42)

    print("\nTraining the model...")
    start_train = time.time()
    model.fit(X_train, y_train)
    print(f"Logistic Regression train time: {time.time() - start_train:.2f}s")

    # 6. Evaluate the model using our universal function
    evaluate_model(model, X_test, y_test)

    # find best model parameters using GridSearchCV

    param_grid = {
        'penalty': ['l2', 'l1', 'elasticnet', None],
        'alpha': [0.0001, 0.001, 0.01],
        'eta0': [0.1, 1.0, 10.0],
        'max_iter': [500, 1000, 2000],
        'fit_intercept': [True, False]
    }


    perceptron = Perceptron(random_state=42)

    grid_search = GridSearchCV(perceptron, param_grid, cv=5, scoring='accuracy')
    grid_search.fit(X_train, y_train)

    # Best model
    best_model = grid_search.best_estimator_
    print("Best parameters:", grid_search.best_params_)

    # Evaluate on test set
    y_pred = best_model.predict(X_test)
    print("Test accuracy:", accuracy_score(y_test, y_pred))
    evaluate_model(best_model, X_test, y_test, model_name="Best Perceptron Classifier")

    

# Run the new function
train_Perceptron_model(X_sampled, y_sampled)

so we see the perceptron is not realy adequate for such a complex task

Sources:
- https://www.kaggle.com/code/jocelyndumlao/birdclef-2025-mfcc-feature-roc-auc-analysis#Import-Libraries (24.05.2025)
- https://github.com/UtrechtUniversity/animal-sounds (24.05.2025)
- https://www.kaggle.com/competitions/birdclef-2025/discussion/572928 (24.05.2025)
- https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Perceptron.html (23.06.2025)